In [1]:
import pandas as pd
import os
from pathlib import Path
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey, Numeric
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship

In [2]:
Base = declarative_base()

# ==================== DEFINIÇÃO DAS TABELAS ====================

class Canal(Base):
    __tablename__ = 'canais'
    ID_Canal = Column(Integer, primary_key=True)
    Descricao_Canal = Column(String(50), nullable=False)

class Categoria(Base):
    __tablename__ = 'categorias'
    ID_Categoria = Column(Integer, primary_key=True)
    Categoria = Column(String(50), nullable=False)
    subcategorias = relationship("Subcategoria", back_populates="categoria")

class Marca(Base):
    __tablename__ = 'marcas'
    ID_Marca = Column(Integer, primary_key=True)
    Marca = Column(String(50), nullable=False)
    produtos = relationship("Produto", back_populates="marca")

class Subcategoria(Base):
    __tablename__ = 'subcategorias'
    ID_Categoria = Column(Integer, ForeignKey('categorias.ID_Categoria'))
    ID_Subcategoria = Column(Integer, primary_key=True)
    Subcategoria = Column(String(50), nullable=False)
    categoria = relationship("Categoria", back_populates="subcategorias")
    produtos = relationship("Produto", back_populates="subcategoria")

class Produto(Base):
    __tablename__ = 'produtos'
    ID_Subcategoria = Column(Integer, ForeignKey('subcategorias.ID_Subcategoria'))
    ID_Produto = Column(Integer, primary_key=True)
    Descricao_Produto = Column(String(100), nullable=False)
    ID_Marca = Column(Integer, ForeignKey('marcas.ID_Marca'))
    Preco_Unitario = Column(Numeric(10, 2))
    Tributos = Column(Numeric(10, 2))
    Custo = Column(Numeric(10, 2))
    subcategoria = relationship("Subcategoria", back_populates="produtos")
    marca = relationship("Marca", back_populates="produtos")

# ==================== FUNÇÕES DE IMPORTAÇÃO ====================
diretorio_atual = Path(os.getcwd())

ARQUIVO_CANAL = diretorio_atual.parent / 'planilhas' / 'Canal.xlsx'     
ARQUIVO_CATEGORIA = diretorio_atual.parent / 'planilhas' / 'Categoria.xlsx'
ARQUIVO_MARCA = diretorio_atual.parent / 'planilhas' / 'Marca.xlsx'      
ARQUIVO_SUBCATEGORIA = diretorio_atual.parent / 'planilhas' / 'Subcategoria.xlsx' 
ARQUIVO_PRODUTO = diretorio_atual.parent / 'planilhas' / 'Produto.xlsx'        
    
BANCO_DADOS = diretorio_atual.parent / 'data' / 'DBVendas.db'


C:\Users\pcwin\AppData\Local\Temp\ipykernel_13444\821523515.py:1: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [3]:
def criar_banco(banco_dados=BANCO_DADOS):
    """Cria o banco de dados"""
    engine = create_engine(f'sqlite:///{banco_dados}', echo=False)
    Base.metadata.create_all(engine)
    print(f"✅ Banco '{banco_dados}' criado")
    return engine

def importar_arquivo_excel(engine, arquivo, modelo, nome_tabela):
    """Importa um arquivo Excel para o banco"""
    
    if not os.path.exists(arquivo):
        print(f"   ⚠️  Arquivo não encontrado: {arquivo}")
        return False
    
    print(f"\n📥 Importando: {arquivo}")
    
    try:
        # Ler o Excel
        df = pd.read_excel(arquivo)
        
        # Limpar dados
        for col in df.columns:
            if df[col].dtype == 'object':
                df[col] = df[col].astype(str).str.strip()
                df[col] = df[col].str.replace(',', '.')
        
        print(f"   📊 {len(df)} registros encontrados")
        
        # Importar para o banco
        Session = sessionmaker(bind=engine)
        session = Session()
        
        for _, row in df.iterrows():
            dados = {k: v for k, v in row.to_dict().items() if pd.notna(v)}
            objeto = modelo(**dados)
            session.merge(objeto)
        
        session.commit()
        print(f"   ✅ {len(df)} registros importados para '{nome_tabela}'")
        session.close()
        
        return True
        
    except Exception as e:
        print(f"   ❌ Erro: {str(e)}")
        return False

# ==================== IMPORTAÇÃO PRINCIPAL ====================

def importar_todas_tabelas(banco_dados=BANCO_DADOS):
    """Importa todas as tabelas na ordem correta"""
    
    print("="*60)
    print("🚀 IMPORTANDO TABELAS PARA O BANCO")
    print("="*60)
    
    # Criar banco
    engine = criar_banco(banco_dados)
    
    # Ordem correta de importação (respeitando chaves estrangeiras)
    # 1º - Tabelas independentes
    importar_arquivo_excel(engine, ARQUIVO_CANAL, Canal, 'canais')
    importar_arquivo_excel(engine, ARQUIVO_CATEGORIA, Categoria, 'categorias')
    importar_arquivo_excel(engine, ARQUIVO_MARCA, Marca, 'marcas')
    
    # 2º - Tabela que depende de categoria
    importar_arquivo_excel(engine, ARQUIVO_SUBCATEGORIA, Subcategoria, 'subcategorias')
    
    # 3º - Tabela que depende de subcategoria e marca
    importar_arquivo_excel(engine, ARQUIVO_PRODUTO, Produto, 'produtos')
    
    print("\n" + "="*60)
    print("✅ IMPORTAÇÃO CONCLUÍDA!")
    print("="*60)
    
    return engine

# ==================== CONSULTAS ====================

def verificar_dados(engine):
    """Verifica se os dados foram importados corretamente"""
    
    Session = sessionmaker(bind=engine)
    session = Session()
    
    print("\n📊 VERIFICANDO DADOS IMPORTADOS:")
    print("-"*50)
    
    # Contar registros
    print(f"   Canais:      {session.query(Canal).count()}")
    print(f"   Categorias:  {session.query(Categoria).count()}")
    print(f"   Marcas:      {session.query(Marca).count()}")
    print(f"   Subcategorias: {session.query(Subcategoria).count()}")
    print(f"   Produtos:    {session.query(Produto).count()}")
    
    # Mostrar alguns produtos
    print("\n📋 PRIMEIROS PRODUTOS:")
    print("-"*70)
    
    produtos = session.query(Produto).join(Marca).join(Subcategoria).limit(5).all()
    for p in produtos:
        print(f"   {p.ID_Produto} - {p.Descricao_Produto:<15} | {p.marca.Marca:<10} | R$ {p.Preco_Unitario}")
    
    session.close()

# ==================== EXECUÇÃO ====================

engine = importar_todas_tabelas(BANCO_DADOS)
    
# Verificar os dados
verificar_dados(engine)
    
# Fechar conexão
engine.dispose()
    
print("\n✅ Tudo pronto! Banco de dados 'DBVendas.db' criado com sucesso!")

🚀 IMPORTANDO TABELAS PARA O BANCO
✅ Banco 'C:\Users\pcwin\Documents\Converte_TXT_CSV\data\DBVendas.db' criado
   ⚠️  Arquivo não encontrado: C:\Users\pcwin\Documents\Converte_TXT_CSV\planilhas\Canal.csv

📥 Importando: C:\Users\pcwin\Documents\Converte_TXT_CSV\planilhas\Categoria.xlsx
   📊 7 registros encontrados
   ✅ 7 registros importados para 'categorias'

📥 Importando: C:\Users\pcwin\Documents\Converte_TXT_CSV\planilhas\Marca.xlsx
   📊 21 registros encontrados
   ✅ 21 registros importados para 'marcas'

📥 Importando: C:\Users\pcwin\Documents\Converte_TXT_CSV\planilhas\Subcategoria.xlsx
   📊 22 registros encontrados
   ✅ 22 registros importados para 'subcategorias'

📥 Importando: C:\Users\pcwin\Documents\Converte_TXT_CSV\planilhas\Produto.xlsx
   📊 213 registros encontrados
   ✅ 213 registros importados para 'produtos'

✅ IMPORTAÇÃO CONCLUÍDA!

📊 VERIFICANDO DADOS IMPORTADOS:
--------------------------------------------------
   Canais:      2
   Categorias:  7
   Marcas:      21
   